The competition's environment plays about **one episode a second**.
Almost every question worth asking about an agent needs hundreds of them,
so that number is what stands between having a hypothesis and having an
answer.

This is how it became **over 24,000 a second** on one
desktop machine, what each step was worth once measured properly, and how
I checked that the faster engine is still the same engine, bit for
bit.

**The short version.** A bit-exact C++ port of the engine already
existed, published by
[@nikital7](https://www.kaggle.com/code/nikital7/4000x-environment-speedup-kaggriculture),
and it is the foundation everything here stands on. I patched it to the
current engine version, then found four more multipliers: one free and
large, three that are pure code with the arithmetic left untouched. The
most useful thing I learned had nothing to do with any of them.

All measurements are on a **Mac mini M4, 24 GB, 10 cores**.

| | episodes per second | versus the official environment |
|---|---|---|
| official `kaggle-environments` | **1.02** | 1x |
| C++ engine, one thread | 4,139 | 4,049x |
| C++ engine, all cores | **24,442** | **23,910x** |

Same two recorded routes playing each other, same seeds, same machine,
same wall clock. The official figure is the median of
20 episodes on `kaggle-environments` 1.32.7.

<a id="s1"></a>
## 1. The measurement that was wrong, and why it comes first

My first pass produced a tidy story: the engine spent 39 % of an episode
in one function, a shortcut halved it, throughput went up sevenfold.
Every one of those numbers was wrong, in the same direction, for the same
reason.

They were measured on a machine running an eight-process search at the
time. Load average 9.9 on 10 cores.

| quantity | measured under load | measured idle | inflation |
|---|---|---|---|
| episode cost | 302 us | 203 us | 1.49x |
| that function's share | 39 % | 24 % | 1.62x |
| the shortcut's speedup | 1.22x | 1.11x | 1.10x |

Those two columns are the same ad-hoc timer run twice, once on a busy
machine and once on a quiet one, which is what makes them a fair pair.
The purpose-built harness in section 5 independently prices the same
pre-fix engine at 203 us, and the interleaved three-engine
benchmark puts it at 205 us. Three instruments, one number.

The tell I nearly missed was the spread rather than the value. 12
paired repetitions of the same two binaries gave ratios from **1.005 to
1.672**. A number that unstable is not a measurement of code, it is a
measurement of what else the machine was doing, and quoting its mean
would have hidden exactly that.

Which is the point the rest of this rests on: **a performance number
without its dispersion is not a result**. Every chart below plots the
individual repetitions.

In [1]:
# The guard both benchmark scripts now open with
import os

def require_idle(max_load=1.5):
    load = os.getloadavg()[0]
    if load > max_load:
        raise SystemExit(
            f"load average {load:.2f} is above {max_load}: throughput "
            f"measured here is not a property of the code")
    return load

print(f"load average on this Kaggle worker: {os.getloadavg()[0]:.2f}")

<a id="s2"></a>
## 2. Methodology, in five rules

Everything after this follows one protocol, and the protocol is doing
most of the work.

**Interleave, do not batch.** Variants are compiled into a single binary
in separate namespaces and timed in rotation, A B C A B C, rather than
all of A then all of B. A machine's clock speed drifts with
temperature, and rotation makes that drift hit every variant equally
instead of accumulating on whichever ran last.

**Report the median of repetitions, and show them all.** Nine
repetitions of 1500 episodes each below. The median
resists one slow run; plotting every point is what lets a reader see
whether the median means anything.

**Pair when comparing two builds.** For a before-and-after, each
repetition measures both, and the statistic is the ratio within the pair
rather than a difference between two independently-taken medians.

**Refuse to measure a loaded machine.** Section 1 is why.

**Check equivalence before reporting speed.** Every throughput figure
here comes from a run that first verified the fast path returns results
identical to the slow one. A faster wrong answer is not a result.

<a id="s3"></a>
## 3. Three engines, nine repetitions each, every point plotted

* **A**, [nikital7's port](https://www.kaggle.com/code/nikital7/4000x-environment-speedup-kaggriculture)
  exactly as published, engine 1.32.6.
* **B**, mine before this iteration: the same code patched to 1.32.7.
* **C**, mine after.

In [2]:
import matplotlib.pyplot as plt

SAMPLES = {'orig_pristine': [202.8, 203.9, 204.1, 204.2, 204.3, 204.7, 205.1, 206.0, 209.9], 'ours_before': [203.2, 203.4, 203.4, 203.6, 204.9, 204.9, 205.1, 205.5, 217.1], 'ours_after': [110.4, 110.7, 110.9, 111.0, 111.0, 111.2, 111.5, 112.4, 116.8]}

labels = ["A  pristine port\n(1.32.6)", "B  patched to 1.32.7\n(before)",
          "C  after this iteration"]
keys = ["orig_pristine", "ours_before", "ours_after"]
colours = ["#9aa3ad", "#2f6fb2", "#14724d"]

fig, ax = plt.subplots(figsize=(9.5, 4.0))
for i, (k, c) in enumerate(zip(keys, colours)):
    v = sorted(SAMPLES[k])
    med = v[len(v) // 2]
    ax.plot([i, i], [v[0], v[-1]], "-", color=c, lw=1.2, alpha=0.45)
    ax.plot([i + (j - len(v) / 2) * 0.045 for j in range(len(v))], v, "o",
            ms=5.5, color=c, alpha=0.6)
    ax.plot([i - 0.2, i + 0.2], [med, med], "-", color=c, lw=3)
    top = max(max(s) for s in SAMPLES.values())
    ax.annotate(f"median {med:.1f} us\n{1e6 / med:,.0f} eps/sec\n"
                f"9 runs: {v[0]:.1f} to {v[-1]:.1f}",
                (i, top * 1.045), fontsize=8.5, color=c, ha="center",
                va="bottom", weight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels(labels, fontsize=9)
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(min(min(s) for s in SAMPLES.values()) * 0.97,
            max(max(s) for s in SAMPLES.values()) * 1.16)
ax.set_ylabel("microseconds per episode\n(lower is better)")
ax.set_title("every repetition plotted, interleaved in one process, "
             "single thread", fontsize=10)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

**A and B overlap**, 204.3 against
204.9 microseconds with ranges that cross. Bringing the
port up to the current engine version, which adds a convex branch to
three price curves, **costs nothing measurable**, and the overlapping
point clouds are how a reader can see that rather than having to trust
it.

**C separates cleanly**, and the useful way to say so is not
"1.84x on the medians against the
port, 1.85x against our own previous
build" but that **its slowest repetition beats the fastest of either other
engine**, 116.8 us against 202.8 and 203.2.
Both ratios appear on this page and they are different comparisons, so
each one names its baseline. Sections 5 and 6 are the two changes behind
it.

<a id="s4"></a>
## 4. The free multiplier: releasing the GIL parallelises nothing

The batch entry point looked like this, and it looked fine:

```cpp
std::vector<std::pair<double, double>> out;
py::gil_scoped_release rel;                    // <- looks like the win
for (const auto& [a, b, seed] : jobs)
    out.push_back(run_episode_raw(*a, *b, seed, steps));
```

Releasing the GIL lets *other Python threads* run while this one works.
It does not make this one use more than a single core, and a caller doing
`run_many(jobs)` has no other Python threads to run. The engine had been
sitting on one core the whole time.

The replacement is a work-stealing pool, and it is safe by construction
rather than by testing: episodes are independent and seeded
independently, job `i` writes only to `out[i]`, the action streams are
`const` and shared read-only, and no RNG state crosses an episode
boundary.

In [3]:
import matplotlib.pyplot as plt

ROWS = [(1, 4139, 1.0, 10.8, [3814, 4124, 4155, 4262]), (2, 8201, 1.98, 1.6, [8112, 8184, 8217, 8243]), (4, 15109, 3.65, 2.0, [14910, 15088, 15129, 15219]), (8, 21696, 5.24, 7.6, [20352, 21449, 21943, 21998]), (0, 24442, 5.9, 15.6, [21624, 24254, 24629, 25439])]
REAL_EPS = 1.022

labels = ["1", "2", "4", "8", "all\ncores"]
med = [r[1] for r in ROWS]
ideal = [med[0] * n for n in (1, 2, 4, 8, 10)]

fig, ax = plt.subplots(figsize=(9.5, 4.0))
ax.plot(range(5), ideal, "--", color="#c9ced4", lw=1.4,
        label="perfect scaling")
for i, r in enumerate(ROWS):
    ax.plot([i] * len(r[4]), r[4], "o", ms=5, color="#2f6fb2", alpha=0.45)
ax.plot(range(5), med, "-", color="#2f6fb2", lw=2, label="median")
for i, (r, m) in enumerate(zip(ROWS, med)):
    ax.annotate(f"{m:,}\n{r[2]:.1f}x", (i, max(r[4])), xytext=(0, 11),
                textcoords="offset points", ha="center", fontsize=9)
ax.axhline(REAL_EPS, color="#b5486b", lw=1.2)
ax.annotate(f"the official environment: {REAL_EPS:.2f}/sec",
            (0.02, REAL_EPS), xytext=(4, 8), textcoords="offset points",
            fontsize=9, color="#b5486b")
ax.set_xticks(range(5)); ax.set_xticklabels(labels)
ax.set_xlabel("worker threads")
ax.set_ylabel("episodes per second")
ax.set_ylim(0, max(ideal) * 1.08)
ax.set_title("what one line of parallelism was worth, all repetitions shown",
             fontsize=10)
ax.legend(frameon=False, fontsize=9, loc="upper left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

Scaling is 5.9x rather than the 10x a core count
suggests, and the spread at the top of the sweep is
16 %. Two honest reasons, and the scatter shows both: ten
cores here are not ten equal cores, and the machine was not exclusively
mine, with a load average of 1.4 during the
sweep. **So this curve is a floor, not a ceiling**, and the wide points
on the right are the machine's other work leaking in rather than
anything about the pool.

<a id="s5"></a>
## 5. How the niches were found, and the first one

Both remaining fixes came from the same procedure, and it is worth
describing because the obvious tool does not work well here. Read the
range column before the cost column: the ablation resolves the two
largest phases and does not resolve the small ones, and a row whose
range is larger than its own value is telling you it has nothing to
say.

**Why not a sampling profiler.** The engine is header-only and compiled
at `-O3`, so nearly everything is inlined. A sampler attributes time to
whatever symbol survived inlining, which on this code is mostly one giant
`step`. The profile it produces is true and useless.

**Ablation instead.** Compile one variant per phase, each with that
phase early-returned, into a single binary in separate namespaces, then
time them all interleaved. The difference from the full engine is what
that phase costs **as the compiler actually built it**, inlining
included.

```cpp
void decay_plants(Farm& f, int step_i) { if (true) return;   // <- stubbed
    ...
}
```

**The caveat, and it is not small:** a stubbed engine computes the wrong
answer. This is a timing instrument only, it never touches a correctness
claim, and the stubbed binaries are throwaway. Any measurement mixing the
two would be worthless.

**Then drill inside the phase that wins.** A phase being expensive does
not say why. Once `end_of_day` came top, timing its pieces separately
(constructing the generator against drawing from it) is what turned "this
function is slow" into "this one line is a constant".

**And the pattern both fixes share, which is the transferable part:**
look for **constants wearing a loop's clothes**, work that is recomputed
identically every time. A scan that always finds nothing on odd steps, a
table that is regenerated on every construction. Neither is a clever
algorithm, both are the same question asked twice: does this have to
happen again?

Run over the idle path as it stood before these fixes, which means a
build checked out at the commit before either of them and given the same
profiling seam, the ablation gives:


| phase | cost per episode | range over 7 passes | share of the measured deltas |
|---|---|---|---|
| **end_of_day** | 140 us | 9 % | **68.6 %** |
| decay_plants | 42 us | 82 % (noise) | 20.6 % |
| town_consume | 15 us | 64 % (noise) | 7.4 % |
| apply_unit_actions | 5 us | 327 % (noise) | 2.3 % |
| process_market | 2 us | 431 % (noise) | 1.0 % |
| | | | |
| summed deltas | 204 us | | 100.0 % |
| residual, the loop itself | -0.9 us | | -0.4 % of the episode |

I had spent my attention on `decay_plants`, which is 21 %. Take it
first anyway, because its fix is the cleaner argument.

`decay_plants` walks every tile of both farms on every turn: 100 tiles,
twice, 719 times, 143,800 visits an episode, looking for plants past
their lifespan. Under an idle stream it finds none, because nothing has
been planted.

The shortcut is a proof rather than a heuristic. Every assignment of a
plant's `max_lifespan_step` has the form `k * turns_per_day`:

```cpp
t.max_lifespan_step = cd.ongoing ? -1
    : (day + cd.max_yield_day + 1) * cfg.turns_per_day;
```

With `turns_per_day` even, which it is at 24, that value is even, so
`(step_i - max_lifespan_step) % 2` is exactly `step_i % 2`. **On an odd
step no tile in the game can decay**, and the scan is provably a no-op:

```cpp
if ((cfg.turns_per_day % 2) == 0 && (step_i % 2) != 0) return;
```

The config guard matters: an odd `turns_per_day` breaks the argument, so
it falls back to scanning rather than silently doing the wrong thing.

**Worth 1.11x**, not the 2x that halving the scans suggests. The naive
model assumes the scan costs what it costs; the branch predictor was
already handling a loop that almost always falls through.

<a id="s6"></a>
## 6. The second niche: a table wearing a loop's clothes

`end_of_day` runs 29 times an episode, once per in-game day: the step
index runs 0 to 718 and the call fires when `(step_i + 1) % 24 == 0`,
which selects step 23 through step 695. Timing its parts says where the
cost goes:


* constructing the Mersenne Twister: **4.92 us**
* one `random()` draw from it: **0.00263 us**


Essentially all initialisation and almost none of it use. The engine
builds a fresh generator for each in-game day, seeded from the episode
seed and the day. The generator is CPython's Mersenne Twister,
reproduced bit for bit by
[nikital7's port](https://www.kaggle.com/code/nikital7/4000x-environment-speedup-kaggriculture),
and its `init_by_array` opens with this:

```cpp
void init_by_array(const uint32_t* key, int keylen) {
    init_genrand(19650218u);        // 624 words, identical every time
    ...                             // two key-dependent mixing passes
}
```

That first line produces **the same 624 words on every construction, in
every episode, forever**. It is a constant table wearing a loop's
clothes. Computed once per process and copied thereafter, the output is
provably identical because it is the same recurrence with the same
constant. Worth 1.13x to 1.20x depending on machine state.

**Then the two mixing passes it opens onto, which are the real bulk.**
They cannot be precomputed, because they depend on the seed. But they
can be written better, and none of that touches the arithmetic:

```cpp
for (; k; --k) {
    mt_[i] = (mt_[i] ^ ((mt_[i-1] ^ (mt_[i-1] >> 30)) * 1664525u)) + key[j] + j;
    ++i; ++j;
    if (i >= N) { mt_[0] = mt_[N-1]; i = 1; }     // true once in 624
    if (j >= keylen) j = 0;                       // keylen is 1 or 2
}
```

Three observations, each removing work that is not the recurrence:

* the wrap test is **true once in 624 iterations**, so the loop is split
  at the wrap rather than tested on every pass;
* `mt_[i-1]` was written by the previous iteration, so it is carried in a
  register instead of reloaded;
* `j` cycles over `keylen`, which for a 64-bit seed is 1 or 2, so those
  two cases lose the counter, its test and its reset entirely.

What remains is a serial dependency chain, each word needing the one
before it, and that is a floor no rewriting goes under. Everything
around it was removable.

In [4]:
import matplotlib.pyplot as plt

PAIRS = [(159.9, 110.8), (160.4, 112.7), (161.6, 113.5), (162.1, 112.8), (159.8, 112.4), (161.9, 113.3), (159.1, 111.7), (158.6, 111.0), (158.9, 112.7)]

before = [p[0] for p in PAIRS]
after = [p[1] for p in PAIRS]
ratios = sorted(b / a for b, a in PAIRS)
med = ratios[len(ratios) // 2]

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.8),
                              gridspec_kw={"width_ratios": [1.4, 1]})
for b, a in PAIRS:
    ax.plot([0, 1], [b, a], "-", color="#c9ced4", lw=1)
ax.plot([0] * len(before), before, "o", ms=6, color="#2f6fb2", alpha=0.65,
        label="before")
ax.plot([1] * len(after), after, "o", ms=6, color="#14724d", alpha=0.65,
        label="after")
ax.set_xticks([0, 1]); ax.set_xticklabels(["before", "after"])
ax.set_xlim(-0.3, 1.3)
ax.set_ylabel("microseconds per episode")
ax.set_title("paired A/B: each line is one repetition\nthat measured both builds",
             fontsize=10)
ax.legend(frameon=False, fontsize=9)

ax2.plot(ratios, range(1, len(ratios) + 1), "o-", ms=5, color="#c2571f")
ax2.axvline(1.0, color="#9aa3ad", ls="--", lw=1)
ax2.set_xlabel("within-pair speedup")
ax2.set_ylabel("repetition (sorted)")
ax2.set_title(f"every pair above 1.0\nmedian {med:.3f}x, "
              f"range {ratios[0]:.3f} to {ratios[-1]:.3f}", fontsize=10)
for a_ in (ax, ax2):
    for s in ("top", "right"):
        a_.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

The right panel is the statistic that matters: **every one of the
nine pairs is above 1.0**, median 1.424x. That is
a stronger claim than a difference of medians, because each pair measured
both builds under the same machine conditions seconds apart.

That chart is the loop rewrite alone, measured on top of the table.
Seeding a generator goes from 4.92 to **2.33 microseconds**, and the two
changes together take the engine from 205 to 111 microseconds an
episode: **1.85x against the port this is built on, single threaded,
with the arithmetic untouched**.

The tightness is worth noting after section 1. 9 pairs spanning 1.410
to 1.443 is a 2.3 % spread; the earlier measurement of the table change
spanned 1.041 to 1.147 on a busier machine. Same protocol, different
amount of interference, and plotting the pairs is what makes the
difference visible instead of averaging it away.

<a id="s7"></a>
## 7. That change adds shared state, so it gets its own proof

The thread pool and the parity shortcut add no shared state. The
precomputed table does: one array, initialised once, read by every
generator in every thread. That is the single thing most likely to break
a deterministic engine, so it gets a test aimed at it rather than a
paragraph of reassurance.

`tools/test_rng_shared.cpp` checks three properties, in a deliberate
order.

**Thread safety first**, in a fresh process, because the race can only be
observed while the static is still uninitialised. Sixteen threads park on
a spin barrier, are released together, and all construct a generator with
the same seed. C++11 and later guarantee that exactly one thread runs a
function-local static's initialiser while the others block; this checks
the guarantee rather than citing it.

**Identity**: the precomputed table must equal `init_genrand(19650218)`
word for word, all 624. Without this the golden tests would tell us
something is wrong, but not what.

**Idempotence**: the same seed must give the same draws no matter what
came before it. A generator built after 5,000 others must match one built
first, and two seeds interleaved in both orders must agree.

These are the results, and they are a table rather than a cell that
prints them, because this notebook cannot run them: they need the C++
toolchain, the trace files and a `kaggle-environments` install that the
page does not carry. A code cell printing hardcoded verdicts would look
like evidence and be typing. Every row below is produced by something in
the repo, named beside it.

| check | scope | verdict | produced by |
|---|---|---|---|
| 16 threads racing the first construction | fresh process | identical sequences | `tools/test_rng_shared.cpp` |
| precomputed table vs `init_genrand(19650218)` | 624 words | identical | `tools/test_rng_shared.cpp` |
| same seed after 5,000 other generators | 32 draws | identical | `tools/test_rng_shared.cpp` |
| two seeds interleaved in both orders | 16 draws each | identical | `tools/test_rng_shared.cpp` |
| golden test through the Python bindings | 6 traces | 6/6 | `tests/test_golden.py` |
| C++ core against the real environment | 6 traces, 719 steps | 6/6 step-exact | `tools/validate.cpp` |
| step-mode observations against the real environment | 6 traces, ~10,000 fields each | 6/6 | `tests/test_l1.py` |
| `run_many` at 1 / 2 / 4 / 8 / auto threads | 3,000 episodes | bit-identical | `research/bench_kagsim.py` |
| **pristine 1.32.6 build on 1.32.7 traces** | **control: must FAIL** | **fails all 6, step 97 to 625** | `tools/validate.cpp` |


The last row is the one I would look at first in somebody else's
speedup. Every other line says "my build agrees with itself". That line
says the harness can tell a wrong engine from a right one, because the
unpatched 1.32.6 build fails every trace the patched build passes. A
validation suite that passes for both is not a validation suite.

<a id="s8"></a>
## 8. What the speed is actually for

Throughput is not the point; it changes which questions are askable.

**Panels stop being anecdotes.** A three-seed evaluation is cheap and
treacherous, because a season's outcome is dominated by the shop draw. At
24,442 episodes a second a 2,000-seed distribution costs a second,
so the question changes from "did it win on my seeds" to "what does its
distribution look like".

**Searches stop being overnight jobs.** A genetic search over schedules
that took hours per generation batch now runs a generation in seconds,
which puts the loop between an idea and its verdict inside a
conversation.

**And negative results get cheap**, which matters more than it sounds.
Most things I try do not work. When a test costs a second the honest
answer is affordable; when it costs an hour there is pressure to keep a
result that looked good early.

<a id="s9"></a>
## 9. Where it lives, what is not built, and an open invitation

Everything here is in one repository: engine, bindings, the six
validation traces, the fidelity harness, the thread-safety test and both
benchmark scripts.
**[github.com/destbreso/kaggriculture-cppsim](https://github.com/destbreso/kaggriculture-cppsim)**
installs straight from git, and the README carries the usage doctrine I
would want a stranger to read first, including the fallback pattern that
degrades to the real environment rather than to a wrong answer.

**What is left, measured rather than guessed.** After all of this,
`end_of_day` is still 62 % of an idle episode, and it is still the same
thing: seeding a generator costs 2.33 microseconds against 0.103 to copy
one that is already seeded, **22 times cheaper**. The engine seeds by
(episode seed, in-game day), so a workload that replays the same seed
panel across many candidates, which is exactly what a search does,
recomputes the same few hundred states over and over. A per-thread cache
of seeded states would take the idle-path episode from 110 to about 45
microseconds, **a further 2.47x**, and it is the one remaining niche I
know of. That base is the idle path measured by the ablation harness,
110.4 us, and the interleaved benchmark on recorded routes
reports 111.0. They agree to within a percent, which they did not
when the ablation was a single pass: taking seven and reporting the
range is what closed the gap, and a disagreement between two of your own
instruments is usually the noisier one telling you so.

It is not in the build, and the reason is the point of section 7. The
three changes above add either no shared state or one read-only table
proven safe. A cache adds mutable shared state whose benefit depends on
how you call the function: for a search replaying 32 seeds it is a large
win, and for the 3,000 distinct seeds this notebook benchmarks it is
pure overhead. That is a different kind of change from the ones here,
and it belongs behind its own flag and its own measurement rather than
smuggled into a page about correctness-preserving speedups.

**The roadmap is open, and one layer is deliberately unbuilt.** L0 is the
batch API this notebook measures, for fixed-versus-fixed work. L1 is step
mode, where a live Python agent plays inside the simulator and receives
the same observation dicts the real interpreter builds. **L2 would be the
vectorised layer**: many episodes advancing in lockstep with tensor
observations, which is what reinforcement learning at scale needs.

L2 is on hold on purpose. It would accelerate nothing that exists today,
since fixed-versus-fixed already saturates in L0 and a live Python agent
bottlenecks on its own Python, which batching the engine cannot fix. And
the hard part of L2 is not the code, it is **the tensor encoding**, which
chosen without the network that will consume it is a representation
chosen blind. So it waits for its first consumer, and the engine side is
a weekend of work once the schema is fixed.

**If you are that consumer I would genuinely like to hear from you**, and
the same goes for anyone with a use for any of this, a suggestion, a
correction, or an interest in building on it. An issue on the repo or a
comment here both work.

**And the credit belongs upstream.** None of this exists without
[@nikital7](https://www.kaggle.com/nikital7)'s
[bit-exact C++ port](https://www.kaggle.com/code/nikital7/4000x-environment-speedup-kaggriculture),
which solved the genuinely hard problems: CPython's Mersenne Twister
exactly, Python dict insertion-order semantics where a shed cap decides
which goods die, and slot-index market resolution. The idea was his. What
I added is a version patch, a thread pool, two small proofs and a
stubborn insistence on measuring an idle machine. If he wants any of it
folded back upstream, it is his.

If you do build on it, the number I would be most curious about is your
scaling curve on hardware with more uniform cores than this one, since
section 4 is measuring a floor.